# Every subpopulation: structure of heterogeneity

Whether a model's spread runs along the survey's real social divides. Per
question, model/variant and mode, over **every retained subpopulation cell of
the run**: the Spearman correlation `rho` between the condensed pairwise-nEMD
vector of the survey cells and that of the model's, plus the Pearson twin.
Both are computed again on each reference country's cells alone — Germany's
144 and Mexico's 131 — and those twins are what the ticks draw: a flat `_` for
Germany, a `/` slash for Mexico.

High `A` with `rho` near zero is the failure this plate is for: a model that
disperses as much as the survey does, but in directions the survey does not
recognise — over-steerability without social fidelity. The corner labels name
all four combinations.

Joined against `population_adaptability.csv` on `["question", "source",
"mode"]` — deliberately not on `model_key`/`arm`, because the two Mixtral
reference rows carry no model key. That notebook must therefore run first.

Every dashed rule here is a **fixed reference, never a fit**: `A = 1` on the x
axis and `correlation = 0` on the y. They never tilt because there is nothing
in them to tilt — they are the yardsticks the points are read against.

| encoding | meaning |
|---|---|
| colour + marker | the model |
| fill | hollow = as released, solid = `german`, bottom-half = `spanish-mx` |
| dotted rule + `_` tick | not a run: the same run over the German cells alone |
| dotted rule + `/` tick | not a run: the same run over the Mexican cells alone |
| which ticks a marker wears | its own target: `german` gets the `_`, `spanish-mx` the `/`, as-released both |
| x | A, model dispersion over survey dispersion |
| y | the survey-to-model correlation of pairwise distances |

Five views per family — every question at once, then happiness, politics,
religion and trust alone. Writes `outputs/culture/population_structure.csv`
plus `figures/fig_population_structure_<view>_{ntp,fa}` and the Pearson
companions, `.png` and `.pdf`.

The p-values are descriptive only: pairwise distances over shared cells are
not independent, so a Mantel-style permutation test would be needed for a real
one. A run that produced only full answers contributes its `fa` rows alone.

`.venv` has no jupyter and a bare `uv run` must not be used here — it destroys
the hand-built sm_120 `llama-cpp-python` wheel. Committed without cell
outputs.

In [1]:
import pandas as pd

from culture.population import (
    ADAPTABILITY_TABLE,
    MODES,
    STRUCTURE_TABLE,
    build_structure,
    check_structure,
)
from culture.population_plates import VIEWS, structure_plate
from culture.tables import read_csv
from machine_bias_reproduction.io_utils import write_csv

## Build

In [2]:
table = build_structure()
check_structure(table)
write_csv(table, STRUCTURE_TABLE)
print(len(table), "rows")

/Users/neemiasbuceli/Documents/phd/TowardsSociallyGroundedAISafety/src/culture/population.py:246: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho = spearmanr(survey, model)
/Users/neemiasbuceli/Documents/phd/TowardsSociallyGroundedAISafety/src/culture/population.py:247: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson = pearsonr(survey, model)


164 rows


## Joining against the adaptability table

In [3]:
adaptability = read_csv(ADAPTABILITY_TABLE)
assert adaptability is not None
joined = table.merge(
    adaptability[["question", "source", "mode", "e_mean_nemd", "adaptability_ratio"]],
    on=["question", "source", "mode"],
)
assert len(joined) == len(table)
print(len(joined), "rows joined")

164 rows joined


## The plates

In [4]:
files = []
for mode in MODES:
    for view in VIEWS:
        files += structure_plate(
            joined, mode, view, "rho_structure", "Spearman rho", "fig_population_structure"
        )
        files += structure_plate(
            joined,
            mode,
            view,
            "pearson_structure",
            "Pearson r",
            "fig_population_structure_pearson",
        )
print(len(files), "files")

40 files


## Reading the plates

Bottom-right is the corner that matters: a model that spreads like the survey
(`A` near 1) while correlating near zero has learned to look heterogeneous
without learning who differs from whom. Top-left is compression that is at
least pointed the right way. Top-right is what real adaptation would look
like.

The `_` tick is the same run over the German cells alone and the `/` tick the
same run over the Mexican cells. A tuned variant whose tick sits above its own
marker tracks that country's social divides better than it tracks the world's;
comparing the as-released variant's tick with the tuned variant's tick of the
same country is the finetuning's effect on that fidelity, which is the question
the adaptability plates cannot answer. A tuned variant carries only its own
country's tick, so the comparison always runs against the as-released marker,
which carries both.

Spearman is the headline because the two distance vectors need only agree on
ordering; the Pearson companions are there to show the conclusion does not
rest on that choice.